In [2]:
# Cell 1: Imports and paths
import pandas as pd
from pathlib import Path

# Notebook lives in notebooks/, so repo root is one level up.
REPO_ROOT = Path.cwd().parent
DATA_DIR = REPO_ROOT / "data"
SCRATCH_FILE = REPO_ROOT / "notebooks" / "nukemap_lookups_template.xlsx"

# Display options
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print(f"Repo root:    {REPO_ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Scratch file: {SCRATCH_FILE}")
print(f"Scratch exists? {SCRATCH_FILE.exists()}")

Repo root:    c:\Users\Vinod\OneDrive\Documents\Masters of Data Science and Innovation (UTS)\36104 Data Visualisation and Narratives\Assignment 3\Bullets-Over-Hugs
Data dir:     c:\Users\Vinod\OneDrive\Documents\Masters of Data Science and Innovation (UTS)\36104 Data Visualisation and Narratives\Assignment 3\Bullets-Over-Hugs\data
Scratch file: c:\Users\Vinod\OneDrive\Documents\Masters of Data Science and Innovation (UTS)\36104 Data Visualisation and Narratives\Assignment 3\Bullets-Over-Hugs\notebooks\nukemap_lookups_template.xlsx
Scratch exists? True


In [4]:
# Cell 2: Load the filled lookup template
# The header is at row 6 (0-indexed: 5) — see template structure.
# We skip rows 0-4 which contain title and run-date metadata.

raw = pd.read_excel(
    SCRATCH_FILE,
    sheet_name="NUKEMAP Lookups",
    header=5,  # row 6 in Excel = index 5 in pandas
)

# Drop any completely-empty rows that might appear after the data
raw = raw.dropna(how="all")

print(f"Shape: {raw.shape}")
print(f"\nColumns: {list(raw.columns)}")
print(f"\nFirst 3 rows:")
display(raw.head(3))

Shape: (7, 18)

Columns: ['display_order', 'bomb_id', 'bomb_name', 'country', 'year', 'yield_kt', 'burst_type', 'centroid_lat', 'centroid_lon', 'description', 'fireball_km', 'heavy_5psi_km', 'moderate_1psi_km', 'thermal_3rd_km', 'fatalities', 'injuries', 'psi_1_population', 'notes']

First 3 rows:


,display_order,bomb_id,bomb_name,country,year,yield_kt,burst_type,centroid_lat,centroid_lon,description,fireball_km,heavy_5psi_km,moderate_1psi_km,thermal_3rd_km,fatalities,injuries,psi_1_population,notes
0,1,little_boy,Little Boy,United States,1945,15,Airburst,-33.8688,151.2093,"Hiroshima atomic bomb. ~140,000 dead by end of...",0.198,1.67,4.52,1.91,17080,82120,259352,Detonation altitude: 600 m. NUKEMAP also repor...
1,2,fat_man,Fat Man,United States,1945,20,Airburst,-33.8688,151.2093,"Nagasaki atomic bomb. ~70,000 dead by end of 1...",0.222,1.72,4.59,2.21,19020,84310,265603,Detonation altitude: 503 m.
2,3,ivy_mike,Ivy Mike,United States,1952,10400,Surface,-33.8688,151.2093,First-ever hydrogen bomb. Vaporised the test i...,3.570,9.99,25.70,29.10,742240,868040,2900155,Surface burst — significant radioactive fallou...


In [5]:
# Cell 2b: Quick sanity confirmation
print(f"Shape: {raw.shape}")
print(f"\nDtypes:")
print(raw.dtypes)
print(f"\nAll bombs in display order:")
print(raw[["display_order", "bomb_name", "yield_kt", "fatalities", "injuries"]].to_string(index=False))

Shape: (7, 18)

Dtypes:
display_order         int64
bomb_id                 str
bomb_name               str
country                 str
year                  int64
yield_kt              int64
burst_type              str
centroid_lat        float64
centroid_lon        float64
description             str
fireball_km         float64
heavy_5psi_km       float64
moderate_1psi_km    float64
thermal_3rd_km      float64
fatalities            int64
injuries              int64
psi_1_population      int64
notes                   str
dtype: object

All bombs in display order:
 display_order    bomb_name  yield_kt  fatalities  injuries
             1   Little Boy        15       17080     82120
             2      Fat Man        20       19020     84310
             3     Ivy Mike     10400      742240    868040
             4 Castle Bravo     15000      889370    944380
             5   Tsar Bomba     50000     1828080   1187490
             6  R-12 (SS-4)      2300      504860   1033750
         

In [6]:
# Cell 3: Validate the loaded data
# These checks catch transcription errors and structural problems before
# they end up in the CSV that the dashboard team consumes.

EXPECTED_BOMBS = 7
EXPECTED_BOMB_IDS = {
    "little_boy", "fat_man", "ivy_mike", "castle_bravo",
    "tsar_bomba", "r12_ss4", "dong_feng_4",
}

# 1. Right number of bombs
assert len(raw) == EXPECTED_BOMBS, f"Expected {EXPECTED_BOMBS} bombs, got {len(raw)}"

# 2. All expected bomb IDs present, no duplicates
assert set(raw["bomb_id"]) == EXPECTED_BOMB_IDS, f"Bomb IDs mismatch: {set(raw['bomb_id']) ^ EXPECTED_BOMB_IDS}"
assert raw["bomb_id"].is_unique, "Duplicate bomb_id detected"

# 3. display_order is 1..7 with no gaps
assert sorted(raw["display_order"].tolist()) == list(range(1, EXPECTED_BOMBS + 1)), \
    f"display_order not 1..7: {sorted(raw['display_order'].tolist())}"

# 4. No nulls anywhere — every cell should be filled
null_counts = raw.isna().sum()
assert null_counts.sum() == 0, f"Nulls found:\n{null_counts[null_counts > 0]}"

# 5. All numeric values positive (no negative radii or casualties)
numeric_cols = ["yield_kt", "fireball_km", "heavy_5psi_km", "moderate_1psi_km",
                "thermal_3rd_km", "fatalities", "injuries", "psi_1_population"]
for col in numeric_cols:
    assert (raw[col] > 0).all(), f"Non-positive value in {col}"

# 6. Centroid is consistent (single Sydney CBD location)
assert raw["centroid_lat"].nunique() == 1, "centroid_lat varies across rows"
assert raw["centroid_lon"].nunique() == 1, "centroid_lon varies across rows"

# 7. Burst type is one of the two valid values
assert set(raw["burst_type"]) <= {"Airburst", "Surface"}, \
    f"Unexpected burst_type values: {set(raw['burst_type'])}"

# 8. Sanity: fatalities scale roughly with yield (allowing for saturation at the top)
# Just check that the smallest yield has fewer fatalities than the largest.
assert raw.loc[raw["yield_kt"].idxmin(), "fatalities"] < raw.loc[raw["yield_kt"].idxmax(), "fatalities"], \
    "Smallest bomb has more fatalities than largest — likely transcription error"

# 9. Sanity: fireball is always the smallest radius
for _, row in raw.iterrows():
    radii = [row["fireball_km"], row["heavy_5psi_km"], row["moderate_1psi_km"], row["thermal_3rd_km"]]
    assert row["fireball_km"] == min(radii), \
        f"Fireball is not the smallest radius for {row['bomb_name']}: {radii}"

print("✓ All validation checks passed.")

✓ All validation checks passed.


In [7]:
# Cell 4: Derive useful fields for the dashboard
# We add a few computed columns that the dashboard team will need.
# Each one earns its keep: it answers a specific dashboard question.

clean = raw.copy()

# Yield in megatons — useful for display when bombs span 4 orders of magnitude
clean["yield_mt"] = (clean["yield_kt"] / 1000).round(2)

# Display label combining name and yield — a tidy dropdown option
# e.g. "Little Boy (15 kt)" or "Tsar Bomba (50 Mt)"
def yield_label(kt):
    if kt < 1000:
        return f"{int(kt)} kt"
    else:
        return f"{kt/1000:.1f} Mt".replace(".0 Mt", " Mt")

clean["yield_display"] = clean["yield_kt"].apply(yield_label)
clean["dropdown_label"] = clean["bomb_name"] + " (" + clean["yield_display"] + ")"

# Total casualties — handy for headline metrics
clean["total_casualties"] = clean["fatalities"] + clean["injuries"]

# Casualty rate per kt — shows the "diminishing returns" at large yields
clean["fatalities_per_kt"] = (clean["fatalities"] / clean["yield_kt"]).round(2)

# Fatality ratio — captures the saturation story (Little Boy: 0.21, Tsar Bomba: 1.54)
clean["fatality_to_injury_ratio"] = (clean["fatalities"] / clean["injuries"]).round(3)

# Reorder columns logically
column_order = [
    # Identifiers + display
    "display_order", "bomb_id", "bomb_name", "dropdown_label",
    # Bomb specs
    "country", "year", "yield_kt", "yield_mt", "yield_display", "burst_type",
    # Geography
    "centroid_lat", "centroid_lon",
    # Effect radii (in inner-to-typical-outer order)
    "fireball_km", "heavy_5psi_km", "thermal_3rd_km", "moderate_1psi_km",
    # Casualties
    "fatalities", "injuries", "total_casualties", "psi_1_population",
    # Derived metrics
    "fatalities_per_kt", "fatality_to_injury_ratio",
    # Context
    "description", "notes",
]
clean = clean[column_order]

# Sort by display_order so CSV is in dropdown order
clean = clean.sort_values("display_order").reset_index(drop=True)

print(f"Shape: {clean.shape}")
print(f"\nDerived fields preview:")
preview_cols = ["bomb_name", "yield_display", "dropdown_label",
                "fatalities_per_kt", "fatality_to_injury_ratio"]
display(clean[preview_cols])

Shape: (7, 24)

Derived fields preview:


,bomb_name,yield_display,dropdown_label,fatalities_per_kt,fatality_to_injury_ratio
0,Little Boy,15 kt,Little Boy (15 kt),1138.67,0.208
1,Fat Man,20 kt,Fat Man (20 kt),951.00,0.226
2,Ivy Mike,10.4 Mt,Ivy Mike (10.4 Mt),71.37,0.855
3,Castle Bravo,15 Mt,Castle Bravo (15 Mt),59.29,0.942
4,Tsar Bomba,50 Mt,Tsar Bomba (50 Mt),36.56,1.539
5,R-12 (SS-4),2.3 Mt,R-12 (SS-4) (2.3 Mt),219.50,0.488
6,Dong Feng-4,3.3 Mt,Dong Feng-4 (3.3 Mt),179.19,0.515


In [8]:
# Cell 5: Write the deliverable CSV to data/

OUTPUT_PATH = DATA_DIR / "nuke_blast_effects.csv"
clean.to_csv(OUTPUT_PATH, index=False)

# Verify by reading it back
df_check = pd.read_csv(OUTPUT_PATH)
size_kb = OUTPUT_PATH.stat().st_size / 1024

print(f"✓ Wrote {OUTPUT_PATH.relative_to(REPO_ROOT)}")
print(f"  Shape: {df_check.shape}")
print(f"  Size:  {size_kb:.1f} KB")
print(f"\nColumns: {list(df_check.columns)}")

✓ Wrote data\nuke_blast_effects.csv
  Shape: (7, 24)
  Size:  2.6 KB

Columns: ['display_order', 'bomb_id', 'bomb_name', 'dropdown_label', 'country', 'year', 'yield_kt', 'yield_mt', 'yield_display', 'burst_type', 'centroid_lat', 'centroid_lon', 'fireball_km', 'heavy_5psi_km', 'thermal_3rd_km', 'moderate_1psi_km', 'fatalities', 'injuries', 'total_casualties', 'psi_1_population', 'fatalities_per_kt', 'fatality_to_injury_ratio', 'description', 'notes']


In [9]:
# Cell 6: Generate the data dictionary

DATA_DICT = """# Nuclear Blast Effects Dataset — Data Dictionary

**Topic 2: Localised Impact Simulator (Sydney CBD)**

## Source

Data collected from [NUKEMAP](https://nuclearsecrecy.com/nukemap/) (Wellerstein, 2012–2026), running each of 7 historical nuclear weapons over Sydney CBD. NUKEMAP's blast and thermal radii come from public-domain physics in:
- Glasstone & Dolan, *The Effects of Nuclear Weapons*, 1977
- Fletcher et al., *Nuclear Bomb Effects Computer*, US Atomic Energy Commission, 1953

Casualty estimates use NUKEMAP's server-side model based on LandScan population data (24-hour ambient population).

**Bomb specifications** (yields, burst altitudes, fission fractions) sourced from NUKEMAP's `presets.js`, traceable to historical record.

## File: `nuke_blast_effects.csv`

One row per bomb. 7 rows × 24 columns.

### Identifiers and display

| Column | Type | Description |
|---|---|---|
| `display_order` | int | 1–7. Recommended dropdown order (chronological by historical first-use). |
| `bomb_id` | str | Machine-friendly key. Use this for joins/lookups. |
| `bomb_name` | str | Display name (e.g. "Little Boy"). |
| `dropdown_label` | str | Pre-built dropdown option text (e.g. "Tsar Bomba (50 Mt)"). |

### Bomb specifications

| Column | Type | Description |
|---|---|---|
| `country` | str | Country of origin/test. |
| `year` | int | Year first tested or deployed. |
| `yield_kt` | int | Yield in kilotons. |
| `yield_mt` | float | Yield in megatons (yield_kt / 1000). |
| `yield_display` | str | Human-readable yield ("15 kt" or "50 Mt"). |
| `burst_type` | str | "Airburst" or "Surface". |

### Geography (Sydney CBD ground zero)

| Column | Type | Description |
|---|---|---|
| `centroid_lat` | float | -33.8688 (Sydney CBD latitude). Same for all rows. |
| `centroid_lon` | float | 151.2093 (Sydney CBD longitude). Same for all rows. |

### Effect radii (km)

These are the four ring radii to draw on the map. **All in kilometres.**

| Column | Type | Description |
|---|---|---|
| `fireball_km` | float | Maximum fireball radius. Inside: vaporisation. |
| `heavy_5psi_km` | float | 5 psi airblast radius. Inside: most residential buildings collapse, fatalities widespread. NUKEMAP labels this "Moderate blast damage"; the brief calls it "Heavy damage". |
| `moderate_1psi_km` | float | 1 psi airblast radius. Inside: glass shatters, light injuries widespread. NUKEMAP labels this "Light blast damage"; the brief calls it "Moderate damage". |
| `thermal_3rd_km` | float | Thermal radiation radius for 3rd-degree burns. Inside: severe burns requiring medical attention. |

⚠ **Ring order is not fixed.** For small bombs (< 10 Mt), thermal sits between the two blast rings. For very large bombs, thermal becomes the outermost ring. Sort rings by radius before drawing — see VISUALISER_GUIDE.md.

### Casualties

| Column | Type | Description |
|---|---|---|
| `fatalities` | int | NUKEMAP estimated fatalities for this bomb over Sydney CBD. |
| `injuries` | int | NUKEMAP estimated injuries. |
| `total_casualties` | int | fatalities + injuries. |
| `psi_1_population` | int | Average people in the 1 psi blast range over a 24-hour period. |

### Derived metrics

| Column | Type | Description |
|---|---|---|
| `fatalities_per_kt` | float | fatalities / yield_kt. Drops sharply with yield (saturation effect). |
| `fatality_to_injury_ratio` | float | fatalities / injuries. Rises with yield — bigger bombs kill rather than injure. |

### Context

| Column | Type | Description |
|---|---|---|
| `description` | str | One-sentence historical context for the bomb. |
| `notes` | str | Per-bomb methodology notes (detonation altitude, NUKEMAP warnings, model failures). |

## ⚠ Methodology caveats (important for narrative integrity)

1. **NUKEMAP's own framing**: *"Modeling casualties from a nuclear attack is difficult. These numbers should be seen as evocative, not definitive."* Display this disclaimer prominently in the dashboard.

2. **Fallout deliberately excluded** per the brief. UCDP/NUKEMAP fatality figures here are direct effects only (blast, thermal, prompt radiation). Real-world fatalities from a surface burst would be much higher when fallout is included — Castle Bravo, Ivy Mike, and Tsar Bomba (the surface bursts here) would produce massive radioactive contamination not reflected in these numbers.

3. **24-hour population averaging**: `psi_1_population` is the average number of people in the 1 psi range over a 24-hour cycle. Daytime CBD population would be much higher; nighttime much lower. The casualty figures use this average.

4. **Yields > 20 Mt are extrapolated**: NUKEMAP explicitly warns that for yields above 20 Mt (Tsar Bomba), the model scales from 20 Mt validation rather than direct calculation. Tsar Bomba figures should be treated as approximate.

5. **Surface vs airburst**: Castle Bravo and Ivy Mike are surface bursts (test detonations on coral atolls). All others are airbursts at altitudes optimised for blast effect. NUKEMAP's defaults for each preset are used.

6. **No 20 psi or 500 rem rings** for some bombs: NUKEMAP cannot compute 20 psi blast or 500 rem radiation effects when the optimal-airblast detonation altitude exceeds the maximum altitude at which those effects reach the ground. Affects R-12, Dong Feng-4, and Tsar Bomba. Not a problem for our dashboard since neither effect is part of the brief.

## Re-running the data preparation

The notebook `notebooks/02_topic2_nuke_simulator.ipynb` produces this file from a hand-collected raw lookup template (`notebooks/nukemap_lookups_template.xlsx`, gitignored). To re-run:

```bash
pip install openpyxl pandas  # in addition to project requirements
```

Open the notebook and run all cells. To regenerate the raw lookups (e.g., if NUKEMAP updates), follow the instructions in the Excel template's "Instructions" sheet.

## Source attribution for academic purposes

When citing the source of the casualty and radius data:

> Wellerstein, Alex. *NUKEMAP*. nuclearsecrecy.com/nukemap/. Lookups performed [INSERT DATE]. Underlying physics from Glasstone & Dolan (1977) and the US AEC Nuclear Bomb Effects Computer (1953).
"""

dict_path = DATA_DIR / "nuke_blast_effects_data_dictionary.md"
dict_path.write_text(DATA_DICT, encoding="utf-8")
print(f"✓ Wrote {dict_path.relative_to(REPO_ROOT)}")
print(f"  Size: {dict_path.stat().st_size / 1024:.1f} KB")

✓ Wrote data\nuke_blast_effects_data_dictionary.md
  Size: 6.1 KB
